# Widget Delivery Case — Analysis

## 1. Cleaning

**What this section does:** loads the raw spreadsheet, profiles every column, removes the junk rows and the
phantom column, parses timestamps, and builds the 10 row-level derived fields defined in `CLAUDE.md`.

**What it feeds:** every later section (reconciliation, scoring, Q2/Q3, sweep, sensitivity) reads the cleaned
`df` built here. Output: `export_01_cleaned.xlsx`.

In [1]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

RAW_PATH = "data/2026_Platform_Ops_Widget_Data.xlsx"
raw = pd.read_excel(RAW_PATH)
print("raw shape:", raw.shape)

raw shape: (1206, 39)


### 1.1 Junk rows and phantom column
The last 6 rows have no `MARKET_NAME`. Their only populated field is `COMPOSITE_SCORE` (the stray 34.5 plus two
other loose values). These look like summary cells sitting under the table, not deliveries.
`Unnamed: 38` is a column with no header. Its single value sits on row DELIVERY_NAME=930.

In [2]:
junk_rows = raw[raw["MARKET_NAME"].isna()]
print("junk rows:", len(junk_rows))
display(junk_rows.dropna(axis=1, how="all"))

print("non-null values in 'Unnamed: 38':")
display(raw.loc[raw["Unnamed: 38"].notna(), ["DELIVERY_NAME", "Unnamed: 38"]])

junk rows: 6


,COMPOSITE_SCORE
1200,NaN
1201,NaN
1202,NaN
1203,34.538563
1204,4.317320
1205,0.539665


non-null values in 'Unnamed: 38':


,DELIVERY_NAME,Unnamed: 38
0,930.0,4.431211


In [3]:
df = raw[raw["MARKET_NAME"].notna()].copy()
df = df.drop(columns=["Unnamed: 38"])
df = df.reset_index(drop=True)

print("clean shape:", df.shape)
print(df["MARKET_NAME"].value_counts())
print("DELIVERY_NAME unique:", df["DELIVERY_NAME"].is_unique)

clean shape: (1200, 38)
MARKET_NAME
Doorlanta    600
Dashattan    600
Name: count, dtype: int64
DELIVERY_NAME unique: True


### 1.2 Column profile (after dropping junk)
One row per column: type, missing count and %, distinct values, and range.

In [4]:
profile_rows = []
for col in df.columns:
    s = df[col]
    row = {
        "column": col,
        "dtype": str(s.dtype),
        "n_missing": s.isna().sum(),
        "pct_missing": round(100 * s.isna().mean(), 1),
        "n_unique": s.nunique(),
        "min": s.dropna().min() if s.notna().any() else None,
        "max": s.dropna().max() if s.notna().any() else None,
    }
    profile_rows.append(row)
column_profile = pd.DataFrame(profile_rows)
column_profile

,column,dtype,n_missing,pct_missing,n_unique,min,max
0,DELIVERY_NAME,float64,0,0.0,1200,1.0,1600.0
1,DELIVERY_RATING,float64,616,51.3,5,1.0,5.0
2,MARKET_NUMBER,float64,0,0.0,2,4001.0,4050.0
3,MARKET_NAME,str,0,0.0,2,Dashattan,Doorlanta
4,TIMEZONE,str,0,0.0,1,Galaxy A,Galaxy A
5,DASHER,str,0,0.0,54,Abhai,Zak
6,IS_FIRST_DELIVERY,float64,0,0.0,2,0.0,1.0
7,VEHICLE,str,0,0.0,3,bicycle,motorcycle
8,CREATED_AT,str,0,0.0,1200,2055-06-01T06:53:46.825653Z,2055-06-13T21:32:39.591796Z
9,QUOTED_DELIVERY_TIME,str,0,0.0,1188,2055-06-01T14:59:15.1Z,2055-06-13T22:45:00Z


In [5]:
# Missing counts by market, for columns that have any missing values
cols_with_missing = [c for c in df.columns if df[c].isna().any()]
df[cols_with_missing].isna().groupby(df["MARKET_NAME"]).sum().T

MARKET_NAME,Dashattan,Doorlanta
DELIVERY_RATING,333,283
DASHER_AT_STORE_TIME,42,82
CANCELLED_AT,600,600
DASHER_TO_RESTAURANT_DURATION,48,84
DASHER_WAIT_DURATION,88,103
RESTAURANT_TO_CUSTOMER_DURATION,0,1
DASHER_TO_STORE_TIME_CATEGORY,173,148
DASHER_AT_STORE_TIME_GALAXY_A,42,82
NUM_ON_TIME_DELIVERIES,415,76


In [6]:
# Value counts for the categorical / flag columns
for col in ["VEHICLE", "ASAP", "FRAUDULENT", "MANUALLY_ASSIGNED", "WAS_BATCHED",
            "IS_FIRST_DELIVERY", "DELIVERY_RATING", "TIMEZONE", "MARKET_NUMBER"]:
    print(col, df[col].value_counts(dropna=False).sort_index().to_dict())

print()
print("Dashers per market:", df.groupby("MARKET_NAME")["DASHER"].nunique().to_dict())

lifetime_cols = ["NUM_DELIVERIES", "COMPOSITE_STAR_RATING", "NUM_FIVE_STARS", "NUM_ON_TIME_DELIVERIES"]
varies = df.groupby(["MARKET_NAME", "DASHER"])[lifetime_cols].nunique(dropna=False) > 1
print("Dashers whose lifetime columns vary across rows:", varies.sum().to_dict())

vehicles_per_dasher = df.groupby(["MARKET_NAME", "DASHER"])["VEHICLE"].nunique()
print("Dashers with >1 vehicle:", vehicles_per_dasher[vehicles_per_dasher > 1].to_dict())

VEHICLE {'bicycle': 369, 'car': 828, 'motorcycle': 3}
ASAP {0.0: 74, 1.0: 1126}
FRAUDULENT {0.0: 1191, 1.0: 9}
MANUALLY_ASSIGNED {0.0: 1175, 1.0: 25}
WAS_BATCHED {0.0: 1043, 1.0: 157}
IS_FIRST_DELIVERY {0.0: 1193, 1.0: 7}
DELIVERY_RATING {1.0: 33, 2.0: 11, 3.0: 25, 4.0: 45, 5.0: 470, nan: 616}
TIMEZONE {'Galaxy A': 1200}
MARKET_NUMBER {4001.0: 600, 4050.0: 600}

Dashers per market: {'Dashattan': 24, 'Doorlanta': 30}
Dashers whose lifetime columns vary across rows: {'NUM_DELIVERIES': 0, 'COMPOSITE_STAR_RATING': 0, 'NUM_FIVE_STARS': 0, 'NUM_ON_TIME_DELIVERIES': 0}
Dashers with >1 vehicle: {('Doorlanta', 'Lauran'): 2}


### 1.3 Parse timestamps
All timestamps are ISO8601 strings. The UTC columns are used for arithmetic between timestamps. The `*_GALAXY_A`
columns hold local time (UTC−4) and are used only for hour-of-day. They also end in `Z`, but the check below shows
they sit exactly 4 hours behind UTC, so the `Z` is a mislabel. Reading `.dt.hour` from them gives the correct
local hour.

In [7]:
UTC_COLS = ["CREATED_AT", "QUOTED_DELIVERY_TIME", "ESTIMATED_DELIVERY_TIME", "ACTUAL_PICKUP_TIME",
            "ACTUAL_DELIVERY_TIME", "DASHER_ASSIGNED_TIME", "DASHER_CONFIRMED_TIME", "DASHER_AT_STORE_TIME"]
LOCAL_COLS = ["ACTUAL_PICKUP_TIME_GALAXY_A", "ACTUAL_DELIVERY_TIME_GALAXY_A",
              "DASHER_ASSIGNED_TIME_GALAXY_A", "DASHER_AT_STORE_TIME_GALAXY_A"]
TIMESTAMP_COLS = UTC_COLS + LOCAL_COLS

for col in TIMESTAMP_COLS:
    df[col] = pd.to_datetime(df[col], format="ISO8601")

# Local offset check: UTC minus local should be exactly 4 hours on every non-null row
for col in LOCAL_COLS:
    utc_col = col.replace("_GALAXY_A", "")
    offset_hours = (df[utc_col] - df[col]).dt.total_seconds() / 3600
    print(col, "UTC - local (hours):", offset_hours.value_counts().to_dict())

# ACTIVE_DATE is the local delivery date
local_delivery_date = df["ACTUAL_DELIVERY_TIME_GALAXY_A"].dt.tz_localize(None).dt.normalize()
print("ACTIVE_DATE != local delivery date:", (df["ACTIVE_DATE"] != local_delivery_date).sum())
print("ACTIVE_DATE range:", df["ACTIVE_DATE"].min().date(), "to", df["ACTIVE_DATE"].max().date())

ACTUAL_PICKUP_TIME_GALAXY_A UTC - local (hours): {4.0: 1200}
ACTUAL_DELIVERY_TIME_GALAXY_A UTC - local (hours): {4.0: 1200}
DASHER_ASSIGNED_TIME_GALAXY_A UTC - local (hours): {4.0: 1200}
DASHER_AT_STORE_TIME_GALAXY_A UTC - local (hours): {4.0: 1076}
ACTIVE_DATE != local delivery date: 0
ACTIVE_DATE range: 2055-06-01 to 2055-06-13


### 1.4 Integrity checks (report only; no rows changed)
Check that the timestamps run in order and that the `*_DURATION` columns (in seconds) agree with the timestamps.

In [8]:
order_pairs = [
    ("CREATED_AT", "DASHER_ASSIGNED_TIME"),
    ("DASHER_ASSIGNED_TIME", "DASHER_CONFIRMED_TIME"),
    ("DASHER_CONFIRMED_TIME", "DASHER_AT_STORE_TIME"),
    ("DASHER_AT_STORE_TIME", "ACTUAL_PICKUP_TIME"),
    ("ACTUAL_PICKUP_TIME", "ACTUAL_DELIVERY_TIME"),
    ("ESTIMATED_DELIVERY_TIME", "QUOTED_DELIVERY_TIME"),
]
order_check = pd.DataFrame(
    [{"earlier": a, "later": b, "rows_out_of_order": (df[a] > df[b]).sum()} for a, b in order_pairs]
)
display(order_check)

def seconds_between(start, end):
    return (df[end] - df[start]).dt.total_seconds()

duration_checks = {
    "CONFIRM_TO_DELIVER_DURATION": seconds_between("DASHER_CONFIRMED_TIME", "ACTUAL_DELIVERY_TIME"),
    "DASHER_TO_RESTAURANT_DURATION": seconds_between("DASHER_CONFIRMED_TIME", "DASHER_AT_STORE_TIME"),
    "DASHER_WAIT_DURATION": seconds_between("DASHER_AT_STORE_TIME", "ACTUAL_PICKUP_TIME"),
    "RESTAURANT_TO_CUSTOMER_DURATION": seconds_between("ACTUAL_PICKUP_TIME", "ACTUAL_DELIVERY_TIME"),
}
dur_rows = []
for col, from_ts in duration_checks.items():
    dur_rows.append({
        "duration_col": col,
        "missing": df[col].isna().sum(),
        "missing_but_timestamps_present": (df[col].isna() & from_ts.notna()).sum(),
        "of_which_negative_from_ts": (df[col].isna() & (from_ts < 0)).sum(),
        "disagree_gt_2s": ((df[col] - from_ts).abs() > 2).sum(),
    })
display(pd.DataFrame(dur_rows))

,earlier,later,rows_out_of_order
0,CREATED_AT,DASHER_ASSIGNED_TIME,0
1,DASHER_ASSIGNED_TIME,DASHER_CONFIRMED_TIME,0
2,DASHER_CONFIRMED_TIME,DASHER_AT_STORE_TIME,8
3,DASHER_AT_STORE_TIME,ACTUAL_PICKUP_TIME,67
4,ACTUAL_PICKUP_TIME,ACTUAL_DELIVERY_TIME,1
5,ESTIMATED_DELIVERY_TIME,QUOTED_DELIVERY_TIME,1


,duration_col,missing,missing_but_timestamps_present,of_which_negative_from_ts,disagree_gt_2s
0,CONFIRM_TO_DELIVER_DURATION,0,0,0,1
1,DASHER_TO_RESTAURANT_DURATION,132,8,8,0
2,DASHER_WAIT_DURATION,191,67,67,0
3,RESTAURANT_TO_CUSTOMER_DURATION,1,1,1,0


In [9]:
# The individual anomalous rows
show = ["DELIVERY_NAME", "MARKET_NAME", "DASHER", "ASAP", "QUOTED_DELIVERY_TIME", "ESTIMATED_DELIVERY_TIME",
        "DASHER_CONFIRMED_TIME", "ACTUAL_PICKUP_TIME", "ACTUAL_DELIVERY_TIME",
        "CONFIRM_TO_DELIVER_DURATION", "RESTAURANT_TO_CUSTOMER_DURATION"]
print("Delivered before pickup:")
display(df.loc[df["ACTUAL_PICKUP_TIME"] > df["ACTUAL_DELIVERY_TIME"], show])
print("Estimate later than quote:")
display(df.loc[df["ESTIMATED_DELIVERY_TIME"] > df["QUOTED_DELIVERY_TIME"], show])
c2d_from_ts = seconds_between("DASHER_CONFIRMED_TIME", "ACTUAL_DELIVERY_TIME")
print("CONFIRM_TO_DELIVER_DURATION disagrees with timestamps:")
display(df.loc[(df["CONFIRM_TO_DELIVER_DURATION"] - c2d_from_ts).abs() > 2, show].assign(c2d_from_ts=c2d_from_ts))
print("IS_FIRST_DELIVERY == 1 (compare lifetime NUM_DELIVERIES):")
display(df.loc[df["IS_FIRST_DELIVERY"] == 1, ["DELIVERY_NAME", "MARKET_NAME", "DASHER", "NUM_DELIVERIES"]])

Delivered before pickup:


,DELIVERY_NAME,MARKET_NAME,DASHER,ASAP,QUOTED_DELIVERY_TIME,ESTIMATED_DELIVERY_TIME,DASHER_CONFIRMED_TIME,ACTUAL_PICKUP_TIME,ACTUAL_DELIVERY_TIME,CONFIRM_TO_DELIVER_DURATION,RESTAURANT_TO_CUSTOMER_DURATION
949,873.0,Doorlanta,Elon,1.0,2055-06-07 21:52:52.103+00:00,2055-06-07 21:32:20.103+00:00,2055-06-07 20:59:11.331781+00:00,2055-06-07 21:29:15.715107+00:00,2055-06-07 21:28:23.637590+00:00,1752.0,NaN


Estimate later than quote:


,DELIVERY_NAME,MARKET_NAME,DASHER,ASAP,QUOTED_DELIVERY_TIME,ESTIMATED_DELIVERY_TIME,DASHER_CONFIRMED_TIME,ACTUAL_PICKUP_TIME,ACTUAL_DELIVERY_TIME,CONFIRM_TO_DELIVER_DURATION,RESTAURANT_TO_CUSTOMER_DURATION
628,200.0,Doorlanta,Jagar,1.0,2055-06-02 23:49:58.539460+00:00,2055-06-02 23:50:58.539460+00:00,2055-06-02 23:12:37.158532+00:00,2055-06-02 23:18:50.154271+00:00,2055-06-02 23:29:34.358639+00:00,1017.0,644.0


CONFIRM_TO_DELIVER_DURATION disagrees with timestamps:


,DELIVERY_NAME,MARKET_NAME,DASHER,ASAP,QUOTED_DELIVERY_TIME,ESTIMATED_DELIVERY_TIME,DASHER_CONFIRMED_TIME,ACTUAL_PICKUP_TIME,ACTUAL_DELIVERY_TIME,CONFIRM_TO_DELIVER_DURATION,RESTAURANT_TO_CUSTOMER_DURATION,c2d_from_ts
785,1339.0,Dashattan,KC,1.0,2055-06-12 00:07:51.648+00:00,2055-06-11 23:47:44.648+00:00,2055-06-11 23:22:31.669049+00:00,2055-06-11 23:41:41.150079+00:00,2055-06-11 23:56:07.826347+00:00,2055.0,866.0,2016.157298


IS_FIRST_DELIVERY == 1 (compare lifetime NUM_DELIVERIES):


,DELIVERY_NAME,MARKET_NAME,DASHER,NUM_DELIVERIES
10,613.0,Doorlanta,Kev,144.0
386,1220.0,Dashattan,Awstin,1403.0
513,729.0,Doorlanta,Jarry,198.0
610,118.0,Doorlanta,Matthew,223.0
618,1554.0,Dashattan,Dave,142.0
622,1111.0,Dashattan,Mike,736.0
1154,1141.0,Dashattan,Barry,418.0


### 1.5 Row-level derived fields
Definitions are copied from `CLAUDE.md` (metrics 1–10) and must not be varied. All timestamp arithmetic uses the
UTC columns; `hour_local` is the one field that reads a `*_GALAXY_A` (local, UTC−4) column.

**Source columns used**

| Raw column | Meaning | Unit |
|---|---|---|
| `CREATED_AT` | Order placed | UTC timestamp |
| `DASHER_ASSIGNED_TIME` | Dasher assigned to the order | UTC timestamp |
| `ESTIMATED_DELIVERY_TIME` | System's raw delivery estimate | UTC timestamp |
| `QUOTED_DELIVERY_TIME` | Delivery time promised to the customer | UTC timestamp |
| `ACTUAL_DELIVERY_TIME` | Delivery completed | UTC timestamp |
| `ACTUAL_DELIVERY_TIME_GALAXY_A` | Same instant, local time | local timestamp |
| `DASHER_AT_STORE_TIME` | Dasher tapped "arrived at store" (blank on 124 rows) | UTC timestamp |
| `DASHER_WAIT_DURATION` | Time from store arrival to pickup (blank on 191 rows) | **seconds** |
| `ASAP` | 1 = on-demand order, 0 = scheduled order | flag |

**Derived fields**

| # | Field | Definition | Formula | Unit / type | Notes |
|---|---|---|---|---|---|
| 1 | `late_min` | Minutes delivered after the quoted time (negative = early) | `(ACTUAL_DELIVERY_TIME − QUOTED_DELIVERY_TIME) / 60 s` | minutes | Measured against the **quoted** promise, never the estimate |
| 1 | `is_late` | Raw late flag | `late_min > 0` | bool | Strict: exactly on time is not late |
| 2 | `wait_min` | Merchant wait at the store | `DASHER_WAIT_DURATION / 60` | minutes | Missing stays NaN (15.9% of rows). Never imputed |
| 3 | `excess_wait` | Wait beyond the 10-minute merchant allowance | `max(wait_min − 10, 0)`; NaN → 0 | minutes | The **only** fillna in the analysis: missing wait = zero forgiveness |
| 4 | `late_min_adj` | Lateness after forgiving excess merchant wait | `late_min − excess_wait` | minutes | Attributes long merchant holds to the merchant, not the Dasher |
| 4 | `is_late_adj` | Adjusted late flag | `late_min_adj > 0` | bool | Basis of `ontime_adj` in the Dasher score |
| 5 | `severe` | Severe lateness | `late_min_adj > 15` | bool | On the **adjusted** number. Basis of `severe_rate` in the score |
| 6 | `assign_lag_min` | Dispatch delay: order created → Dasher assigned | `(DASHER_ASSIGNED_TIME − CREATED_AT) / 60 s` | minutes | Computed on all rows; meaningful for ASAP only (scheduled orders are assigned ahead of time by design) |
| 7 | `buffer_min` | Cushion the quote adds on top of the raw estimate | `(QUOTED_DELIVERY_TIME − ESTIMATED_DELIVERY_TIME) / 60 s` | minutes | Expect 0 for scheduled, ~17 for ASAP |
| 8 | `late_vs_estimate` | Late against the raw estimate instead of the quote | `ACTUAL_DELIVERY_TIME > ESTIMATED_DELIVERY_TIME` | bool | Used only to show the quote is the fair benchmark (ASAP ≈ 56%) |
| 9 | `hour_local` | Local hour of delivery, 0–23 | `ACTUAL_DELIVERY_TIME_GALAXY_A.dt.hour` | int | For hour-of-day analysis (Q2 demand peaks) |
| 10 | `arrival_logged` | Dasher logged a store arrival | `DASHER_AT_STORE_TIME.notna()` | bool | Basis of `arrival_share` (professionalism) in the score |

Fields 1–5 form a chain: `late_min` → subtract `excess_wait` → `late_min_adj` → thresholds at 0 (`is_late_adj`) and
15 (`severe`). The rest are independent of each other.

In [10]:
def minutes_between(start, end):
    return (df[end] - df[start]).dt.total_seconds() / 60

# 1. Lateness vs the quoted time (strictly greater than 0 = late)
df["late_min"] = minutes_between("QUOTED_DELIVERY_TIME", "ACTUAL_DELIVERY_TIME")
df["is_late"] = df["late_min"] > 0

# 2. Merchant wait in minutes (missing stays missing)
df["wait_min"] = df["DASHER_WAIT_DURATION"] / 60

# 3. Wait beyond 10 minutes is forgiven; missing wait -> 0 forgiveness
df["excess_wait"] = (df["wait_min"] - 10).clip(lower=0).fillna(0)

# 4. Adjusted lateness
df["late_min_adj"] = df["late_min"] - df["excess_wait"]
df["is_late_adj"] = df["late_min_adj"] > 0

# 5. Severe lateness (on the adjusted number)
df["severe"] = df["late_min_adj"] > 15

# 6. Assignment lag (meaningful for ASAP orders only)
df["assign_lag_min"] = minutes_between("CREATED_AT", "DASHER_ASSIGNED_TIME")

# 7. Quote buffer: quoted minus estimated
df["buffer_min"] = minutes_between("ESTIMATED_DELIVERY_TIME", "QUOTED_DELIVERY_TIME")

# 8. Late against the raw estimate
df["late_vs_estimate"] = df["ACTUAL_DELIVERY_TIME"] > df["ESTIMATED_DELIVERY_TIME"]

# 9. Local hour of delivery
df["hour_local"] = df["ACTUAL_DELIVERY_TIME_GALAXY_A"].dt.hour

# 10. Did the Dasher log arrival at the store?
df["arrival_logged"] = df["DASHER_AT_STORE_TIME"].notna()

DERIVED = ["late_min", "is_late", "wait_min", "excess_wait", "late_min_adj", "is_late_adj", "severe",
           "assign_lag_min", "buffer_min", "late_vs_estimate", "hour_local", "arrival_logged"]
df[DERIVED].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
late_min,1200.0,NaN,NaN,NaN,-12.6152,18.222834,-61.879956,-24.375999,-14.813403,-3.961245,179.426294
is_late,1200,2,False,990,NaN,NaN,NaN,NaN,NaN,NaN,NaN
wait_min,1009.0,NaN,NaN,NaN,10.036455,9.729333,0.016667,3.466667,6.966667,13.733333,76.45
excess_wait,1200.0,NaN,NaN,NaN,2.949917,7.177276,0.0,0.0,0.0,1.633333,66.45
late_min_adj,1200.0,NaN,NaN,NaN,-15.565117,17.838054,-61.879956,-26.663741,-17.836475,-7.128004,179.426294
is_late_adj,1200,2,False,1039,NaN,NaN,NaN,NaN,NaN,NaN,NaN
severe,1200,2,False,1149,NaN,NaN,NaN,NaN,NaN,NaN,NaN
assign_lag_min,1200.0,NaN,NaN,NaN,20.104292,178.977265,0.044579,0.362908,4.637724,9.544172,5666.781894
buffer_min,1200.0,NaN,NaN,NaN,16.310097,7.343573,-1.0,10.383333,18.883333,20.725,33.55
late_vs_estimate,1200,2,True,646,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# Sanity checks against CLAUDE.md's stated expectations
print("wait_min missing share:", round(df["wait_min"].isna().mean(), 3), "(expect ~16%)")
print("buffer_min mean by ASAP:", df.groupby("ASAP")["buffer_min"].mean().round(2).to_dict(),
      "(expect 0 scheduled, ~17 ASAP)")
print("late_vs_estimate mean, ASAP only:", round(df.loc[df["ASAP"] == 1, "late_vs_estimate"].mean(), 3),
      "(expect ~0.56)")

wait_min missing share: 0.159 (expect ~16%)
buffer_min mean by ASAP: {0.0: 0.0, 1.0: 17.38} (expect 0 scheduled, ~17 ASAP)
late_vs_estimate mean, ASAP only: 0.556 (expect ~0.56)


### 1.6 Hand-check: DELIVERY_NAME = 930
This row is scheduled (ASAP=0), has a missing wait, and has quoted == estimated. Its values are read straight from
the workbook with openpyxl, independently of pandas, and the derived fields are recomputed by hand.

In [12]:
wb = load_workbook(RAW_PATH, read_only=True)
ws = wb.active
header = [cell.value for cell in next(ws.iter_rows(min_row=1, max_row=1))]
raw_930 = None
for row in ws.iter_rows(min_row=2, values_only=True):
    if row[header.index("DELIVERY_NAME")] == 930:
        raw_930 = dict(zip(header, row))
        break
wb.close()

check_fields = ["ASAP", "DASHER", "MARKET_NAME", "QUOTED_DELIVERY_TIME", "ESTIMATED_DELIVERY_TIME",
                "ACTUAL_DELIVERY_TIME", "CREATED_AT", "DASHER_ASSIGNED_TIME", "DASHER_WAIT_DURATION",
                "DASHER_AT_STORE_TIME", "ACTUAL_DELIVERY_TIME_GALAXY_A"]
{f: raw_930[f] for f in check_fields}

{'ASAP': 0,
 'DASHER': 'Katelynn',
 'MARKET_NAME': 'Doorlanta',
 'QUOTED_DELIVERY_TIME': '2055-06-08T16:30:00Z',
 'ESTIMATED_DELIVERY_TIME': '2055-06-08T16:30:00Z',
 'ACTUAL_DELIVERY_TIME': '2055-06-08T16:43:26.092195Z',
 'CREATED_AT': '2055-06-08T14:40:31.776158Z',
 'DASHER_ASSIGNED_TIME': '2055-06-08T15:24:32.128072Z',
 'DASHER_WAIT_DURATION': None,
 'DASHER_AT_STORE_TIME': None,
 'ACTUAL_DELIVERY_TIME_GALAXY_A': '2055-06-08T12:43:26.092195Z'}

In [13]:
# Hand calculation from the raw strings above:
#   quoted   = 16:30:00 UTC, estimated = 16:30:00 UTC  -> buffer 0, scheduled order
#   actual   = 16:43:26.09 UTC                         -> late by 13.43 min
#   wait     = missing                                 -> excess_wait 0, adjusted lateness = 13.43 min
#   13.43 is not > 15                                  -> not severe
#   created 14:40:31.78, assigned 15:24:32.13          -> assign lag 44.01 min
#   local delivery 12:43                               -> hour_local 12
#   DASHER_AT_STORE_TIME missing                       -> arrival_logged False
row_930 = df.loc[df["DELIVERY_NAME"] == 930].iloc[0]
expected_930 = {
    "ASAP": 0, "late_min": 13.435, "is_late": True, "wait_min": np.nan, "excess_wait": 0.0,
    "late_min_adj": 13.435, "is_late_adj": True, "severe": False, "assign_lag_min": 44.006,
    "buffer_min": 0.0, "late_vs_estimate": True, "hour_local": 12, "arrival_logged": False,
}
check_930 = pd.DataFrame({
    "hand_calc": pd.Series(expected_930),
    "pipeline": row_930[list(expected_930)],
})
check_930["match"] = [
    (pd.isna(h) and pd.isna(p)) or (isinstance(h, float) and abs(h - p) < 0.01) or (h == p)
    for h, p in zip(check_930["hand_calc"], check_930["pipeline"])
]
display(check_930)
assert check_930["match"].all(), "row 930 hand-check failed"
print("DELIVERY_NAME 930 hand-check: PASS")

,hand_calc,pipeline,match
ASAP,0,0.0,True
late_min,13.435,13.43487,True
is_late,True,True,True
wait_min,NaN,NaN,True
excess_wait,0.0,0.0,True
late_min_adj,13.435,13.43487,True
is_late_adj,True,True,True
severe,False,False,True
assign_lag_min,44.006,44.005865,True
buffer_min,0.0,0.0,True


DELIVERY_NAME 930 hand-check: PASS


### 1.7 Acceptance checks

In [14]:
assert len(df) == 1200, len(df)
assert (df["MARKET_NAME"].value_counts() == 600).all()
assert not any(c.startswith("Unnamed") for c in df.columns)
REQUIRED_DERIVED = ["late_min", "wait_min", "excess_wait", "late_min_adj", "is_late_adj", "severe",
                    "assign_lag_min", "buffer_min", "late_vs_estimate", "hour_local", "arrival_logged"]
missing_derived = [c for c in REQUIRED_DERIVED if c not in df.columns]
assert not missing_derived, missing_derived
print("Phase 1 acceptance: PASS —", len(df), "rows,", df.shape[1], "columns")

Phase 1 acceptance: PASS — 1200 rows, 50 columns


### 1.8 Cleaning log
| # | Decision | Rows / cols affected | Basis |
|---|---|---|---|
| 1 | Dropped 6 trailing rows with no `MARKET_NAME`. Their only data is `COMPOSITE_SCORE` (34.54, 4.32, 0.54). They are loose cells, not deliveries | 6 rows | CLAUDE.md dataset facts |
| 2 | Dropped `Unnamed: 38`, a column with no header. Its only value is 4.43 on DELIVERY_NAME 930 | 1 col | CLAUDE.md dataset facts |
| 3 | Parsed 12 timestamp columns as ISO8601. UTC columns are used for arithmetic; the `*_GALAXY_A` columns, verified as UTC−4 on every row, are used for hour-of-day. Their `Z` suffix is a mislabel | 12 cols | CLAUDE.md dataset facts; offset check §1.3 |
| 4 | Converted durations from seconds to minutes when deriving (`wait_min`). The raw `*_DURATION` columns stay in seconds | — | CLAUDE.md |
| 5 | `DASHER_WAIT_DURATION` missing on 191 rows (15.9%) and kept as NaN. 124 of them have no store arrival logged. The other 67 have `DASHER_AT_STORE_TIME` *after* pickup (by 0–13 min), so the source left the wait blank instead of storing a negative | 191 rows | CLAUDE.md: never impute |
| 6 | `excess_wait` fills NaN wait with 0 (missing wait = zero forgiveness). This is the only fillna | 191 rows | CLAUDE.md metric 3 |
| 7 | Kept DELIVERY_NAME 873 (Doorlanta, Elon). Its delivery timestamp is 52 s *before* pickup, so `RESTAURANT_TO_CUSTOMER_DURATION` is NaN. The metrics in scope don't use pickup→delivery, so nothing changes | 1 row | report only |
| 8 | Kept DELIVERY_NAME 200 (Doorlanta, Jagar, ASAP). Its estimate is 1 min later than the quote, giving buffer_min = −1.0 | 1 row | report only |
| 9 | Kept DELIVERY_NAME 1339 (Dashattan, KC). `CONFIRM_TO_DELIVER_DURATION` = 2055 s, but the timestamps give 2016 s (a 39 s gap). Metrics use timestamps, not this column | 1 row | report only |
| 10 | 8 rows have store arrival before confirmation, so `DASHER_TO_RESTAURANT_DURATION` is NaN while the timestamps exist. That column isn't used in phase 1 metrics | 8 rows | report only |
| 11 | `IS_FIRST_DELIVERY = 1` on 7 rows, yet those Dashers have 142–1,403 lifetime deliveries. The flag looks unreliable; checked in phase 4 | 7 rows | report only |
| 12 | Lauran (Doorlanta) appears with 2 vehicles. Vehicle is treated per delivery | 1 Dasher | CLAUDE.md |
| 13 | Lifetime columns are constant within every Dasher (verified) | — | supports `.last()` later |
| 14 | Columns excluded by design: `COMPOSITE_SCORE`, `NUM_ON_TIME_DELIVERIES` (40.9% missing), `DASHER_TO_STORE_TIME_CATEGORY` (26.8% missing). `CANCELLED_AT` is 100% empty. All are kept in the export, used in no metric | 4 cols | CLAUDE.md |
| 15 | `DELIVERY_RATING` missing on 616 rows (51.3%): those deliveries were not rated. Left as NaN | 616 rows | rated-only denominator later |
| 16 | `assign_lag_min`, `buffer_min` and `late_vs_estimate` are computed on all rows. Filtering to ASAP happens at the point of use | — | CLAUDE.md metrics 6–8 |
| 17 | Data years read 2055, a synthetic date shift. Only month and day are used | — | report only |

### 1.9 Export
Timezones are stripped before writing, because Excel can't store tz-aware datetimes.

In [15]:
export = df.copy()
for col in TIMESTAMP_COLS:
    export[col] = export[col].dt.tz_localize(None)
export.to_excel("export_01_cleaned.xlsx", sheet_name="cleaned_deliveries", index=False)
print("wrote export_01_cleaned.xlsx:", export.shape)

wrote export_01_cleaned.xlsx: (1200, 50)


## 2. Verification

## 3. Market comparison

## 4. Measurement

## 5. Dasher scoring

## 6. Q2 sizing

## 7. Q3 baselines

## 8. Exploratory sweep

## 9. Sensitivity